In [5]:
import os
from dotenv import load_dotenv
load_dotenv()



True

In [6]:
if os.environ.get("API_KEY") is None:
    raise ValueError("API_KEY is not Set")
else:
    print ("API_KEY is Set")

API_KEY is Set


In [7]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model= "gpt-5-nano", temperature=0, api_key = os.environ.get("API_KEY")
)

llm.invoke("give me a detailed reason as to why Bleach is the best anime of all time")

AIMessage(content='Here’s a detailed case for why many fans consider Bleach the best anime of all time. It’s subjective, of course, but the series has several core strengths that stand out when you compare it to other big titles.\n\n- Rich, expansive world-building\n  - Bleach delivers a fully realized extra-world: the Human World, Soul Society, and Hueco Mundo each have distinct cultures, rules, and aesthetics. The Gotei 13, the Captain/Visored dynamics, and the hierarchy of the Soul Society give the story a sense of scale and history that many shonen only hint at.\n  - The lore around Zanpakuto—its spirit, Shikai, and Bankai forms—adds a unique layer to combat and character identity. Each weapon isn’t just a tool but a character with its own personality and history.\n\n- Deep, evolving power system\n  - The progression from Shikai to Bankai creates meaningful milestones for characters and keeps battles fresh. The idea that a weapon has a personality and can evolve mirrors real charac

### RAG IMPLEMENTATION WITH YOUR OWN TEXT DATA

STEP 1: Preparing Document for your Text

In [8]:
from langchain_core.documents import Document

my_text = """ Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals.[1]

High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, play and analysis in strategy games (e.g., chess and Go), and content generation (e.g., text, images, audio, and videos).

The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics.[a] To reach these goals, AI researchers use techniques including state space search and mathematical optimisation, formal logic, artificial neural networks, and methods based on statistics, operations research, and economics.[b] AI also draws upon psychology, linguistics, philosophy, neuroscience, and other fields.[2] Some companies, such as OpenAI, Google DeepMind, and Meta, aim to create artificial general intelligence (AGI)—AI that can complete nearly any cognitive task at least as well as a human.[3]

Artificial intelligence was founded as an academic discipline in 1956.[4] The field went through multiple cycles of optimism throughout its history,[5][6] followed by periods of disappointment and loss of funding, known as AI winters.[7][8] Funding and interest increased substantially after 2012, when graphics processing units (GPUs) started being used to accelerate neural networks, and deep learning outperformed previous AI techniques.[9] This growth accelerated further after 2017 with the transformer architecture.[10] In the 2020s, an AI boom coincided with advances in generative AI, which became widespread and allowed for the creation and modification of media. In addition to AI safety and unintended consequences and harms from the use of AI, ethical concerns, AI's long-term effects, environmental effects,[11] and potential existential risks have prompted discussions of AI regulation.

"""

docs = [Document(page_content=my_text, metadata={"source": "my_text.txt"})]

docs

[Document(metadata={'source': 'my_text.txt'}, page_content=" Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals.[1]\n\nHigh-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, play and analysis in strategy games (e.g., chess and Go), and content generation (e.g., text, images, audio, and videos).\n\nThe traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics.[a] To reach th

### Step 2 SPlitting The Documents Into Chunks

In [ ]:
import sys
print(sys.executable)

/Users/jessepepple/Downloads/RAG_Project_Main/.venv/bin/python


In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

chunks = splitter.split_documents(docs)
chunks





[Document(metadata={'source': 'my_text.txt'}, page_content='Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals.[1]'),
 Document(metadata={'source': 'my_text.txt'}, page_content='High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, play and analysis in strategy games (e.g., chess and Go), and content generation (e.g., text, images, audio, and videos).'),
 Document(metadata={'source': 'my_text.txt'}, page_content='The traditional goals of AI research include learning, reasoning, knowledge

###  Step 3 Create Embeddings For The Chunks

In [14]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(model= "text-embedding-3-small", api_key = os.environ.get("API_KEY"))

### Step 4 Create and Store Embeddings in Vector Store

In [19]:
from langchain_community.vectorstores import Chroma

vectore_store = Chroma.from_documents(documents=chunks, embedding=embedding_model)

### Step 5 Similarity Search

In [22]:
vectore_store.similarity_search("What is AI?", k=2)

[Document(metadata={'source': 'my_text.txt'}, page_content='Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals.[1]'),
 Document(metadata={'source': 'my_text.txt'}, page_content='Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their envi

In [23]:
context = vectore_store.similarity_search("What is AI?", k=2)

Talk to LLM

In [25]:
response  = llm.invoke(f"What is AI? Using the following context : {context}")

print(response.content)

Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence—such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops methods and software enabling machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.
